In [16]:
import pandas as pd

# Pick some data file...
df = pd.read_csv('./restaurant-ratings/midwest/east-north-central.csv')
df2 = pd.read_csv('./restaurant-ratings/midwest/east-north-central.csv')

In [17]:
df = pd.concat([df, df2], ignore_index=True)

In [18]:
df.count()

rating    27239366
text      27239030
type      27239366
dtype: int64

In [19]:
# Thank you Gemini
def proportional_sample(df, col_name, sample_size):
    """
    Samples a DataFrame proportionally to the distribution of values in a column,
    without replacement.

    Args:
        df (pd.DataFrame): The DataFrame to sample from.
        col_name (str): The name of the column to use for proportional sampling.
        sample_size (int): The desired size of the sampled DataFrame.

    Returns:
        pd.DataFrame: The sampled DataFrame.
    """

    value_counts = df[col_name].value_counts(normalize=True)
    sampled_counts = (value_counts * sample_size).round().astype(int)

    sampled_dfs = []
    for value, count in sampled_counts.items():
        subset = df[df[col_name] == value].sample(n=min(count, len(df[df[col_name] == value])), replace=False) #sample without replacement, take the minimum of the count or the amount of that value in the df.
        sampled_dfs.append(subset)

    sampled_df = pd.concat(sampled_dfs)
    return sampled_df


sample_size = 24000000
df_train = proportional_sample(df, 'rating', sample_size)

In [20]:
sample_indices = df_train.index
original_indices = df.index

# Find the indices of rows *not* in the sample
rows_to_keep_indices = original_indices.difference(sample_indices)

# Filter the original DataFrame
df_remaining = df.loc[rows_to_keep_indices]
df_test = proportional_sample(df_remaining, 'rating', 3000000)

In [21]:
print('train')
print(df_train.groupby('rating').count())
print('total:',  df_train.count())
print()

print('test')
print(df_test.groupby('rating').count())
print('total:', df_test.count())


train
            text      type
rating                    
1        1798478   1798507
2        1216653   1216682
3        2369659   2369736
4        5070956   5071004
5       13543958  13544070
total: rating    23999999
text      23999704
type      23999999
dtype: int64

test
           text     type
rating                  
1        224808   224813
2        152082   152085
3        296208   296217
4        633868   633876
5       1692995  1693009
total: rating    3000000
text      2999961
type      3000000
dtype: int64


In [23]:
# Save the datafiles
df_test.to_csv('./splits/midwest-big-test.csv', index=False)
df_train.to_csv('./splits/midwest-big-train.csv', index=False)